## Checking GPU


In [ ]:
!nvidia-smi

Installing YOLO through ultralytics


In [ ]:
pip install ultralytics

## Downloading Lablled data from ROBOFLOW


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("lr02").project("braintumor-pzusy")
version = project.version(2)
dataset = version.download("yolov8")
dataset = version.download("yolov8")


Training tumor detection model using yolov8


In [ ]:
!yolo task=detect mode=train model=yolov8s.pt data=/content/brainTumor-2/data.yaml epochs=30 imgsz=650

Confusion matrix


In [ ]:
Image(filename=f'/content/drive/MyDrive/braintumor/runs/detect/train2/confusion_matrix_normalized.png', width=600)

Results


In [ ]:
Image(filename=f'/content/drive/MyDrive/braintumor/runs/detect/train2/results.png', width=800)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
results = pd.read_csv('./runs/detect/train2/results.csv')

In [ ]:
results.columns

In [ ]:
import matplotlib.pyplot as plt

epochs = results['                  epoch']
mAP50_values = results['       metrics/mAP50(B)']
precision_values = results['   metrics/precision(B)']


fig, ax1 = plt.subplots()


color = 'tab:red'
ax1.set_xlabel('Epochs')
ax1.set_ylabel('mAP50', color=color)
ax1.plot(epochs, mAP50_values, color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Precision', color=color)
ax2.plot(epochs, precision_values, color=color)
ax2.tick_params(axis='y', labelcolor=color)


plt.title('mAP50 and Precision Over Epochs')
plt.show()


Cloning ESRGAN model for image enhancement


In [ ]:
# prompt: generate ESRGAN  model and save the model

!git clone https://github.com/xinntao/ESRGAN  



In [ ]:
ls

In [ ]:
%cd ESRGAN

In [ ]:
ls

In [ ]:
!python test.py

## Making predictions for UNSEEN data


In [ ]:
from ultralytics import YOLO

In [ ]:

model = YOLO('./runs/detect/train2/weights/best.pt')


results = model(['./samples/images.jpeg'])

# Process results list
for result in results:
    boxes = result.boxes
    masks = result.masks
    keypoints = result.keypoints
    probs = result.probs
    result.show()
    result.save(filename='result.jpg')


In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO('yolov8n.pt')  # load an official model
model = YOLO('./runs/detect/train2/weights/best.pt')  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map    # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  

In [ ]:
from IPython.display import Image

In [ ]:
Image(filename=f'./runs/detect/val2/confusion_matrix.png', width=800)

In [ ]:
Image(filename=f'./runs/detect/val2/F1_curve.png', width=800)

In [ ]:
Image(filename=f'./runs/detect/val2/P_curve.png', width=800)

In [ ]:
import os
import csv

# Assuming you have already imported the necessary libraries and defined the YOLO model class

# Define a function to process images in a directory
def process_images_in_directory(directory_path, output_directory):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    
    # Initialize CSV file for storing prediction results
    csv_file = open(os.path.join(output_directory, 'predictions.csv'), 'w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['Image', 'Box', 'Mask', 'Keypoints', 'Probs'])  # Writing header
    
    # Iterate over images in the directory
    for filename in os.listdir(directory_path):
        if filename.endswith(('.jpg', '.jpeg', '.png')):  # Adjust extensions as needed
            image_path = os.path.join(directory_path, filename)
            
            # Perform inference on the image
            results = model([image_path])

            # Process results list
            for idx, result in enumerate(results):
                boxes = result.boxes
                masks = result.masks
                keypoints = result.keypoints
                probs = result.probs
                
                # Save the result image
                result_image_path = os.path.join(output_directory, f'{filename[:-4]}_{idx}.jpg')  # Output image path
                result.save(filename=result_image_path)

                # Write prediction results to CSV
                csv_writer.writerow([filename, boxes, masks, keypoints, probs])
                
                # Calculate accuracy (example calculation, modify as per your requirement)
                if result.probs is not None:
                    accuracy = calculate_accuracy(result)
                    print(f'Accuracy for {filename}_{idx}: {accuracy}')
    
    # Close the CSV file after writing
    csv_file.close()

# Function to calculate accuracy (example function, replace with your own calculation)
def calculate_accuracy(result):
    # Example calculation
    return sum(result.probs) / len(result.probs)

# Usage example
directory_path = './brainTumor-2/test/images'
output_directory = './output'
process_images_in_directory(directory_path, output_directory)
